# 4.5 - SmartReplay

Keep 15 exemplars per class and mix them into every training batch.

Compare against 4.2 and 4.4, which cost nothing.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "xx"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Preprocess data**

In [ ]:
# BASE
cfg.OUTPUT_ROOT = cfg.BASE_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.SELECTED_CLASSES)

# TASK 1
cfg.OUTPUT_ROOT = cfg.TASK1_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.TASK_1)

# TASK 2
cfg.OUTPUT_ROOT = cfg.TASK2_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.TASK_2)

In [ ]:
cfg.SELECTED_CLASSES

In [ ]:
print(cfg.SELECTED_CLASSES + cfg.TASK_1)

In [ ]:
# The cache fixes the label space, so it is built once here for all tasks.
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2

class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

print(f"\nclass_to_idx:\n{class_to_idx}")

## **Create Loaders**

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)
check_cache_labels(META, ALL_CLASSES_LIST)

TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2"]

base_test_loader  = emb_loaders(CACHE, [0], "test")[1]
task1_test_loader = emb_loaders(CACHE, [1], "test")[1]
task2_test_loader = emb_loaders(CACHE, [2], "test")[1]

combined_test_loader  = emb_loaders(CACHE, [0, 1], "test")[1]
combined_test_loader2 = emb_loaders(CACHE, [0, 1, 2], "test")[1]

base_train,  base_y   = task_split(CACHE, 0, "train")
base_val,    base_vy  = task_split(CACHE, 0, "val")
task1_train, task1_y  = task_split(CACHE, 1, "train")
task1_val,   task1_vy = task_split(CACHE, 1, "val")
task2_train, task2_y  = task_split(CACHE, 2, "train")
task2_val,   task2_vy = task_split(CACHE, 2, "val")

print(f"base  test {len(base_test_loader.dataset)}")
print(f"task1 test {len(task1_test_loader.dataset)}")
print(f"task2 test {len(task2_test_loader.dataset)}")

## **Upload Model**

In [ ]:
ARM_KEY  = "smart_replay"
ARM_NAME = "SmartReplay"
ARM_CONFIG = {"mechanism": "replay", "per_class": 15, "strategy": "hard"}

CL_EPOCHS, CL_LR, CL_WD, CL_BATCH, CL_LS = 15, 5e-4, 0.03, 32, 0.10

head_base = load_or_train_base_head(
    f"{cfg.CKPT_DIR}/head_base.pt", CACHE, LSTM_STATE, len(cfg.SELECTED_CLASSES), device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH, label_smoothing=CL_LS,
)

In [ ]:
buffer = SmartReplayBuffer(max_per_class=15)
fill_buffer(buffer, base_train, base_y, cfg.SELECTED_CLASSES, class_to_idx, "hard")

head_task1 = expand_head(copy.deepcopy(head_base), len(cfg.SELECTED_CLASSES) + len(cfg.TASK_1))
model_task1 = ModelWrapper(head_task1).to(device)

In [ ]:
results_baseline = evaluate_all_tasks(
    model            = model_task1,
    device           = device,
    base_loader      = base_test_loader,
    task_loaders     = [task1_test_loader],
    combined_loaders = [combined_test_loader],
)

In [ ]:
print(f"head_task1 classifier: in={head_task1.classifier.in_features}  out={head_task1.classifier.out_features}")

## **Task One**

In [ ]:
set_seed(cfg.SEED)

head_task1, train_task1 = train_task(
    head_task1, task1_train, task1_y, task1_val, task1_vy, device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH, label_smoothing=CL_LS,
    buffer=buffer,
    tag=ARM_KEY,
)
model_task1 = ModelWrapper(head_task1).to(device)

In [ ]:
plot_training_results(train_task1)

In [ ]:
results_after_t1 = evaluate_all_tasks(
    model            = model_task1,
    device           = device,
    base_loader      = base_test_loader,
    task_loaders     = [task1_test_loader],
    combined_loaders = [combined_test_loader],
)

In [ ]:
_, all_preds, all_labels = test_model(model_task1, combined_test_loader, device)

plot_confusion_matrix(
    all_labels, all_preds,
    num_classes  = len(cfg.SELECTED_CLASSES) + len(cfg.TASK_1),
    classes_list = ALL_CLASSES_LIST,
    name         = f"Task 1 - {ARM_NAME}",
)

In [ ]:
print_detailed_metrics(
    all_labels   = all_labels,
    all_preds    = all_preds,
    num_classes  = len(cfg.SELECTED_CLASSES) + len(cfg.TASK_1),
    classes_list = ALL_CLASSES_LIST,
    split_old    = (0, len(cfg.SELECTED_CLASSES)),
    split_new    = (len(cfg.SELECTED_CLASSES), len(cfg.SELECTED_CLASSES) + len(cfg.TASK_1)),
)

## **Task 2**

In [ ]:
fill_buffer(buffer, task1_train, task1_y, cfg.TASK_1, class_to_idx, "hard")


head_task2 = expand_head(head_task1, len(ALL_CLASSES_LIST))
model_task2 = ModelWrapper(head_task2).to(device)

In [ ]:
results_baseline = evaluate_all_tasks(
    model            = model_task2,
    device           = device,
    base_loader      = base_test_loader,
    task_loaders     = [task1_test_loader, task2_test_loader],
    combined_loaders = [combined_test_loader, combined_test_loader2],
)

In [ ]:
set_seed(cfg.SEED)

head_task2, train_task2 = train_task(
    head_task2, task2_train, task2_y, task2_val, task2_vy, device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH, label_smoothing=CL_LS,
    buffer=buffer,
    tag=ARM_KEY,
)
model_task2 = ModelWrapper(head_task2).to(device)

In [ ]:
plot_training_results(train_task2)

In [ ]:
results_after_t2 = evaluate_all_tasks(
    model            = model_task2,
    device           = device,
    base_loader      = base_test_loader,
    task_loaders     = [task1_test_loader, task2_test_loader],
    combined_loaders = [combined_test_loader, combined_test_loader2],
)

In [ ]:
_, all_preds, all_labels = test_model(model_task2, combined_test_loader2, device)

plot_confusion_matrix(
    all_labels, all_preds,
    num_classes  = len(ALL_CLASSES_LIST),
    classes_list = ALL_CLASSES_LIST,
    name         = f"Task 2 - {ARM_NAME}",
)

In [ ]:
print_detailed_metrics(
    all_labels   = all_labels,
    all_preds    = all_preds,
    num_classes  = len(ALL_CLASSES_LIST),
    classes_list = ALL_CLASSES_LIST,
    split_old    = (0, len(cfg.SELECTED_CLASSES)),
    split_new    = (len(cfg.SELECTED_CLASSES), len(ALL_CLASSES_LIST)),
)

## **Latent Space**

In [ ]:
preds_emb, labels_emb = predict_head(head_task2, CACHE, [0, 1, 2], device)

res_emb = report_accuracy(preds_emb, labels_emb, TASK_CLASSES, TASK_NAMES, class_to_idx,
                          title=f"FINAL - EMBEDDINGS ({ARM_NAME})")

In [ ]:
xy, lab = project_head_space(head_task2, CACHE, [0, 1, 2], device, seed=cfg.SEED)
task_of = {class_to_idx[c]: t for t, g in enumerate(TASK_CLASSES) for c in g}

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plot_latent_space(xy, lab, ALL_CLASSES_LIST, task_of=task_of,
                  title="Latent space by task", ax=ax[0])
plot_latent_space(xy, lab, ALL_CLASSES_LIST,
                  title="Latent space by class", ax=ax[1])
plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/latent_{ARM_KEY}.png", dpi=150, bbox_inches="tight")
plt.show()

## **Evaluation on Real Images**

In [ ]:
if not clips_available(TASK_ROOTS, "test"):
    print("Raw clips not found. Re-run the Preprocess cell to evaluate on real images.")
    res_real = None
else:
    backbone = load_frozen_backbone(
        device, ResNet50LSTMTeacher, cfg.RESNET50_PATH,
        hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES),
        dropout_p=cfg.DROPOUT_P, remap_fn=remap_teacher_checkpoint,
    )
    full_model = FullModel(backbone, head_task2, backbone_dim=cfg.BACKBONE_DIM).to(device)

    real_loader = clip_loaders(TASK_ROOTS, TASK_CLASSES, class_to_idx, "test",
                               batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS)[1]
    preds_real, labels_real = predict(full_model, real_loader, device)

    res_real = report_accuracy(preds_real, labels_real, TASK_CLASSES, TASK_NAMES,
                               class_to_idx, title=f"FINAL - REAL IMAGES ({ARM_NAME})")

In [ ]:
if res_real is not None:
    plot_confusion_matrix(
        labels_real, preds_real,
        num_classes  = len(ALL_CLASSES_LIST),
        classes_list = ALL_CLASSES_LIST,
        name         = f"Real images - {ARM_NAME}",
    )

    compare_embeddings_vs_real(res_emb, res_real, TASK_NAMES)

## **Save**

In [ ]:
ROWS = [
    {0: eval_head(head_base, *task_split(CACHE, 0, "test"), device)},
    eval_tasks_upto(head_task1, CACHE, 1, device),
    eval_tasks_upto(head_task2, CACHE, 2, device),
]
BYTES_PER_CLASS = buffer_bytes_per_class(buffer, 15) if "buffer" in dir() else 0

R, m = save_arm_result(f"{cfg.RESULTS_DIR}/stage1_arm_{ARM_KEY}.json",
                       ARM_NAME, ROWS, TASK_NAMES, BYTES_PER_CLASS,
                       config=ARM_CONFIG, ceiling=None)
cl_report(R, ARM_NAME, TASK_NAMES)
torch.save(head_task2.state_dict(), f"{cfg.CKPT_DIR}/head_{ARM_KEY}.pt")

with open(f"{cfg.RESULTS_DIR}/stage1_final_{ARM_KEY}.json", "w") as f:
    json.dump({"embeddings": res_emb, "real_images": res_real}, f, indent=2, default=float)